In [1]:
!pip install -q -U "transformers>=4.45.0,<5.0.0" huggingface_hub datasets peft trl bitsandbytes accelerate wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 41.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [2]:
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN"))
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shreyashgaur01 (Nyaya-LLM) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import wandb
wandb.init(
    entity="Nyaya-LLM",
    project="nyaya-llm-phase1",
    id="jafoi0on",
    resume="must",
)
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "google/gemma-3-4b-it"
DATA_PATH = "/kaggle/input/datasets/shreyashgaur222/nyaya-llm-dataset/nyaya_phase2_mixed_train.jsonl"
VAL_PATH = "/kaggle/input/datasets/shreyashgaur222/nyaya-llm-dataset/all_acts_val.jsonl"
OUTPUT_DIR = "/kaggle/working/results/qlora_phase2_gemma3_4b"
MAX_SEQ_LENGTH = 1024

RESUMING = True
CHECKPOINT_PATH = "/kaggle/input/datasets/shreyashgaur222/gemma-qlora/results/qlora_phase2_gemma3_4b/checkpoint-2960"

def format_prompt(example):
    return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

class GemmaSFTTrainer(SFTTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        if "token_type_ids" not in inputs:
            inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])
        return super().training_step(model, inputs, num_items_in_batch)

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        if "token_type_ids" not in inputs:
            inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys)

def main():
    print("Loading dataset...")
    dataset = load_dataset("json", data_files={"train": DATA_PATH, "validation": VAL_PATH})

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
    tokenizer.pad_token = tokenizer.eos_token

    print("Configuring 4-bit Quantization (QLoRA)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    print("Loading base model in 4-bit...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=False,
        dtype=torch.float16,
    )

    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False

    if RESUMING:
        print(f"Loading saved LoRA adapter from checkpoint: {CHECKPOINT_PATH}")
        model = PeftModel.from_pretrained(
            model,
            CHECKPOINT_PATH,
            is_trainable=True,
        )
    else:
        print("Initializing fresh LoRA adapter...")
        peft_config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules="all-linear",
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, peft_config)

    model.print_trainable_parameters()

    print("Setting up Trainer...")
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=25,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=False,
        bf16=False,
        optim="paged_adamw_8bit",
        save_strategy="steps",
        save_steps=740,      
        save_total_limit=5,
        max_length=MAX_SEQ_LENGTH,
        report_to="wandb",
        dataloader_pin_memory=False,
    )

    trainer = GemmaSFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        formatting_func=format_prompt,
        args=training_args,
    )

    if RESUMING:
        print(f"Resuming training from {CHECKPOINT_PATH}...")
        trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)
    else:
        print("Starting fresh training...")
        trainer.train()

    print("Saving model adapter...")
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Gemma-3-4B QLoRA Phase 2 Training Complete.")

if __name__ == "__main__":
    main()

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260301_092835-jafoi0on
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run phase2-qlora-gemma3-4b
wandb: ⭐️ View project at https://wandb.ai/Nyaya-LLM/nyaya-llm-phase1
wandb: 🚀 View run at https://wandb.ai/Nyaya-LLM/nyaya-llm-phase1/runs/jafoi0on
2026-03-01 09:28:54.788728: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772357334.994468      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772357335.052404      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772357335.515334      24 computation_placer.cc:177]

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loading tokenizer...


...

Truncating eval dataset:   0%|          | 0/969 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Resuming training from /kaggle/input/datasets/shreyashgaur222/gemma-qlora/results/qlora_phase2_gemma3_4b/checkpoint-2960...


Step,Training Loss,Validation Loss
3000,0.285000,0.389560
3100,0.285000,0.372675
3200,0.266800,0.366326
3300,0.295800,0.362556
3400,0.298900,0.360716
3500,0.271300,0.355785
3600,0.264400,0.351845
3700,0.288000,0.350084


Saving model adapter...
Gemma-3-4B QLoRA Phase 2 Training Complete.
